# F1 Tyre Degradation — FastF1 Data Download (Colab)

Standalone copy of `src/data/download.py` from the `f1-tyre-degradation` project, adapted to run here so you can pull data from a different network in parallel with any run happening on your own machine (FastF1's public mirror rate-limits at **500 calls/hour**, almost certainly per network/IP — running from Colab gets its own separate budget).

**Storage**: saves to Colab's local notebook storage (`/content/data/raw`), **not** Google Drive — which means it's **ephemeral**: it's wiped if the runtime disconnects or resets. There's a zip-and-download cell near the end — run it periodically (every hour or so during a long run) to pull a snapshot down to your own computer as a safety net, and definitely run it once at the end before closing the tab.

**How to use this:**
1. Run every cell top to bottom (`Runtime > Run all`), skipping the zip/download cell until you actually want a snapshot (it's not part of the automatic flow).
2. If the runtime disconnects, anything not yet zipped and downloaded is gone — re-running from scratch means re-downloading everything, since there's no persistent state to resume from the way there is with Drive. Zip-and-download often if a long run matters to you.
3. When you have a zip on your computer, unzip it and copy `data/raw/*` into your local `f1-tyre-degradation/data/raw/` — the folder layout (`data/raw/<year>/r<round>_<code>/`) is identical, so it's a plain merge, no special handling needed.

This notebook only handles **Phase 1 (data acquisition)** — nothing here trains a model or needs GPU. CPU-only Colab runtime is fine (`Runtime > Change runtime type > CPU`).

In [ ]:
!pip install -q fastf1 pandas pyarrow

In [ ]:
import time, json, csv, logging
from pathlib import Path

import pandas as pd
import fastf1

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger("f1_download")

# ---- config (mirrors config/default.yaml's `data:` block) ----
# Split across two parallel downloaders to multiply effective throughput past
# FastF1's 500-calls/hour-per-network limit: Colab takes 2024-2026, the local
# machine takes 2022-2023. No overlap, so no wasted duplicate work -- merge
# both halves' data/raw folders together afterwards.
SEASONS = [2024, 2025, 2026]

# Tyre "eras" -- the exact mid-2023 Pirelli construction-change race could not be
# confirmed from available sources, so this uses the simpler, defensible split:
# a full calendar year per era rather than an unconfirmed mid-season boundary.
ERA_BOUNDARIES = {
    "GE1": (2022, 2022),
    "GE2": (2023, 2025),
    "R26": (2026, 2026),
}

# fastf1.get_testing_session has no enumeration API -- probe this bounded grid;
# a nonexistent (test_number, session_number) raises ValueError immediately.
TESTING_PROBE = {"test_numbers": [1, 2, 3], "session_numbers": [1, 2, 3]}

# Everything lives under Colab's LOCAL notebook storage -- ephemeral, wiped on
# disconnect/reset. Zip-and-download periodically (see the cell near the end).
CACHE_DIR = Path("/content/fastf1_cache")
DATA_ROOT = Path("/content/data")
RAW_DIR = DATA_ROOT / "raw"
ERRORS_CSV = DATA_ROOT / "download_errors.csv"
SUMMARY_CSV = DATA_ROOT / "download_summary.csv"
RATE_LIMIT_SLEEP_S = 1.0

CACHE_DIR.mkdir(parents=True, exist_ok=True)
RAW_DIR.mkdir(parents=True, exist_ok=True)
fastf1.Cache.enable_cache(str(CACHE_DIR))

# FastF1's underlying data providers cap requests at 500/hour (~7-8 calls per
# session). Once exhausted, EVERY call fails instantly -- without backoff, a
# long run burns through its whole remaining schedule in seconds, logging
# hundreds of false "failures" that are really rate-limit hits.
RATE_LIMIT_BACKOFF_S = 300.0
RATE_LIMIT_MAX_RETRIES = 20  # 20 * 300s = 100 min, comfortably longer than the 1h window

# Generic transient-failure retry, separate from and much shorter than the
# rate-limit backoff above.
GENERIC_RETRY_ATTEMPTS = 3
GENERIC_RETRY_BASE_S = 1.0

SESSION_NAME_TO_CODE = {
    "Practice 1": "FP1",
    "Practice 2": "FP2",
    "Practice 3": "FP3",
    "Qualifying": "Q",
    "Sprint Qualifying": "SQ",
    "Sprint Shootout": "SQ",  # 2023 naming for the same session
    "Sprint": "SPRINT",
    "Race": "R",
}

print("Config ready. Output going to:", RAW_DIR)

In [ ]:
# ---- helpers (ported from src/data/download.py) ----

def era_for(year: int) -> str:
    for name, (start, end) in ERA_BOUNDARIES.items():
        if start <= year <= end:
            return name
    raise ValueError(f"no era configured for year {year}")


def _is_rate_limit_error(exc: Exception) -> bool:
    return type(exc).__name__ == "RateLimitExceededError" or "RateLimitExceeded" in str(type(exc))


def _call_with_retries(fn, *, what: str):
    """Two layers of retry: a ValueError (\"doesn't exist\") is never retried;
    other exceptions get 3 short exponential backoffs; a rate-limit error
    escalates to the long 300s/20-retry wait."""
    for rl_attempt in range(1, RATE_LIMIT_MAX_RETRIES + 1):
        rate_limited = False
        for attempt in range(1, GENERIC_RETRY_ATTEMPTS + 1):
            try:
                return fn()
            except ValueError:
                raise
            except Exception as exc:
                if _is_rate_limit_error(exc):
                    rate_limited = True
                    break
                if attempt == GENERIC_RETRY_ATTEMPTS:
                    raise
                backoff = GENERIC_RETRY_BASE_S * (2 ** (attempt - 1))
                logger.warning(
                    "transient error while %s (attempt %d/%d): %s -- retrying in %.0fs",
                    what, attempt, GENERIC_RETRY_ATTEMPTS, exc, backoff,
                )
                time.sleep(backoff)
        if rate_limited:
            logger.warning(
                "rate limit hit while %s (rate-limit attempt %d/%d) -- sleeping %.0fs",
                what, rl_attempt, RATE_LIMIT_MAX_RETRIES, RATE_LIMIT_BACKOFF_S,
            )
            time.sleep(RATE_LIMIT_BACKOFF_S)
            continue
        raise RuntimeError(f"unreachable retry state while {what}")
    raise RuntimeError(f"rate limit still exceeded after {RATE_LIMIT_MAX_RETRIES} retries: {what}")


def _session_dir(year: int, round_number: int, session_code: str) -> Path:
    return RAW_DIR / f"{year}" / f"r{round_number:02d}_{session_code}"


def _already_downloaded(session_dir: Path) -> bool:
    return (session_dir / "laps.parquet").exists() and (session_dir / "meta.json").exists()


def _log_error(year, event, session, error):
    ERRORS_CSV.parent.mkdir(parents=True, exist_ok=True)
    is_new = not ERRORS_CSV.exists()
    with open(ERRORS_CSV, "a", newline="") as f:
        writer = csv.writer(f)
        if is_new:
            writer.writerow(["year", "event", "session", "error_type", "error_message"])
        writer.writerow([year, event, session, type(error).__name__, str(error)])


def _persist_session(session, out_dir: Path, *, year, event_name, round_number,
                      session_code, era, session_type) -> int:
    out_dir.mkdir(parents=True, exist_ok=True)

    laps = session.laps.copy()
    for col in laps.columns:
        if laps[col].dtype == "object":
            laps[col] = laps[col].astype(str)
    laps.to_parquet(out_dir / "laps.parquet")

    weather = session.weather_data.copy()
    weather.to_parquet(out_dir / "weather.parquet")

    meta = {
        "year": year, "event": event_name, "round": round_number, "session": session_code,
        "era": era, "session_type": session_type,
        "total_laps": int(len(laps)), "n_drivers": int(laps["Driver"].nunique()) if len(laps) else 0,
    }
    (out_dir / "meta.json").write_text(json.dumps(meta, indent=2))
    return len(laps)


def _enumerate_event_sessions(event: pd.Series):
    names = []
    for i in range(1, 6):
        name = event.get(f"Session{i}")
        if isinstance(name, str) and name:
            names.append(name)
    return names

print("Helpers ready.")

In [ ]:
# ---- the two download passes ----

def download_regular_sessions(seasons):
    rows = []
    for year in seasons:
        era = era_for(year)
        try:
            schedule = _call_with_retries(
                lambda: fastf1.get_event_schedule(year, include_testing=False),
                what=f"fetching {year} schedule",
            )
        except Exception as exc:
            logger.error("could not fetch schedule for %s: %s", year, exc)
            _log_error(year, "SCHEDULE", "-", exc)
            continue

        for _, event in schedule.iterrows():
            round_number = int(event["RoundNumber"])
            event_name = str(event["EventName"])

            for session_name in _enumerate_event_sessions(event):
                session_code = SESSION_NAME_TO_CODE.get(
                    session_name, session_name.upper().replace(" ", "_")
                )
                session_dir = _session_dir(year, round_number, session_code)

                if _already_downloaded(session_dir):
                    rows.append({"year": year, "round": round_number, "event": event_name,
                                 "session": session_code, "era": era,
                                 "outcome": "skipped_cached", "laps": None})
                    continue

                try:
                    def _load(_year=year, _round=round_number, _name=session_name):
                        s = fastf1.get_session(_year, _round, _name)
                        s.load(laps=True, telemetry=False, weather=True, messages=True)
                        return s

                    loaded = _call_with_retries(
                        _load, what=f"{year} {event_name} R{round_number} {session_name}"
                    )
                    n_laps = _persist_session(
                        loaded, session_dir, year=year, event_name=event_name,
                        round_number=round_number, session_code=session_code, era=era,
                        session_type=session_code,
                    )
                    rows.append({"year": year, "round": round_number, "event": event_name,
                                 "session": session_code, "era": era,
                                 "outcome": "downloaded", "laps": n_laps})
                    logger.info("downloaded %s %s R%d %s: %d laps",
                                year, event_name, round_number, session_code, n_laps)
                except Exception as exc:
                    _log_error(year, event_name, session_code, exc)
                    rows.append({"year": year, "round": round_number, "event": event_name,
                                 "session": session_code, "era": era,
                                 "outcome": "failed", "laps": None})
                    logger.warning("failed %s %s R%d %s: %s",
                                   year, event_name, round_number, session_code, exc)

                time.sleep(RATE_LIMIT_SLEEP_S)

    return rows


def download_testing_sessions(seasons):
    rows = []
    test_numbers = TESTING_PROBE["test_numbers"]
    session_numbers = TESTING_PROBE["session_numbers"]

    for year in seasons:
        era = era_for(year)
        for test_number in test_numbers:
            for session_number in session_numbers:
                session_code = f"TEST{test_number}S{session_number}"
                session_dir = _session_dir(year, 0, session_code)

                if _already_downloaded(session_dir):
                    rows.append({"year": year, "round": 0, "event": "Pre-Season Testing",
                                 "session": session_code, "era": era,
                                 "outcome": "skipped_cached", "laps": None})
                    continue

                try:
                    def _load(_year=year, _t=test_number, _s=session_number):
                        s = fastf1.get_testing_session(_year, _t, _s)
                        s.load(laps=True, telemetry=False, weather=True, messages=True)
                        return s

                    loaded = _call_with_retries(
                        _load, what=f"{year} testing test={test_number} session={session_number}"
                    )
                    n_laps = _persist_session(
                        loaded, session_dir, year=year, event_name="Pre-Season Testing",
                        round_number=0, session_code=session_code, era=era,
                        session_type="TESTING",
                    )
                    rows.append({"year": year, "round": 0, "event": "Pre-Season Testing",
                                 "session": session_code, "era": era,
                                 "outcome": "downloaded", "laps": n_laps})
                    logger.info("downloaded %s testing t%ds%d: %d laps",
                                year, test_number, session_number, n_laps)
                except ValueError:
                    rows.append({"year": year, "round": 0, "event": "Pre-Season Testing",
                                 "session": session_code, "era": era,
                                 "outcome": "not_applicable", "laps": None})
                except Exception as exc:
                    _log_error(year, "Pre-Season Testing", session_code, exc)
                    rows.append({"year": year, "round": 0, "event": "Pre-Season Testing",
                                 "session": session_code, "era": era,
                                 "outcome": "failed", "laps": None})
                    logger.warning("failed %s testing t%ds%d: %s",
                                   year, test_number, session_number, exc)

                time.sleep(RATE_LIMIT_SLEEP_S)

    return rows


def print_summary(summary: pd.DataFrame) -> None:
    total = len(summary)
    succeeded = int((summary["outcome"] == "downloaded").sum())
    cached = int((summary["outcome"] == "skipped_cached").sum())
    failed = int((summary["outcome"] == "failed").sum())
    not_applicable = int((summary["outcome"] == "not_applicable").sum())
    total_laps = int(summary["laps"].fillna(0).sum())
    present = summary[summary["outcome"].isin(["downloaded", "skipped_cached"])]

    print("=" * 60)
    print("DOWNLOAD SUMMARY")
    print("=" * 60)
    print(f"sessions attempted     : {total}")
    print(f"  downloaded           : {succeeded}")
    print(f"  already cached       : {cached}")
    print(f"  failed               : {failed}")
    print(f"  testing slot n/a     : {not_applicable}")
    print(f"total laps             : {total_laps}")
    print()
    print("present sessions per era:")
    print(present.groupby("era").size().to_string())
    print()
    print("laps per era:")
    print(present.groupby("era")["laps"].sum().to_string())
    print("=" * 60)

print("Download functions ready.")

## Optional: quick smoke test first

Runs against a tiny slice (2026 only, mostly testing sessions) to confirm everything works in this Colab environment before committing to the full multi-hour run. Safe to skip straight to "Full run" below if you'd rather not wait.

In [ ]:
smoke_rows = download_regular_sessions([2026]) + download_testing_sessions([2026])
print_summary(pd.DataFrame(smoke_rows))

## Full run

This is the multi-hour part. **Remember**: storage here is local to this Colab runtime and is wiped on disconnect/reset — there's no Drive backing it. Use the zip-and-download cell below periodically so a disconnect doesn't cost you the whole run.

If you do get disconnected before zipping, re-running this cell starts over from whatever survived (usually nothing, for local storage) rather than resuming cleanly the way it would with persistent storage.

In [ ]:
all_rows = download_regular_sessions(SEASONS) + download_testing_sessions(SEASONS)
summary = pd.DataFrame(all_rows)
SUMMARY_CSV.parent.mkdir(parents=True, exist_ok=True)
summary.to_csv(SUMMARY_CSV, index=False)
print_summary(summary)

## Checking progress without stopping the download

Open a **second Colab tab on the same notebook** (`Runtime > Manage sessions`, or just duplicate the tab) and run only this cell there — it just reads what's on local disk, no network calls, so it's safe to run while the full-run cell above is still going in the first tab.

In [ ]:
meta_files = list(RAW_DIR.glob("**/meta.json"))
total_laps = sum(json.loads(f.read_text())["total_laps"] for f in meta_files)
print(f"sessions downloaded so far: {len(meta_files)}")
print(f"total laps so far: {total_laps}")

## Zip and download a snapshot

Run this any time you want a safety-net copy on your own computer — including once at the very end. It zips `/content/data` and triggers a browser download. For a large corpus this zip can be sizeable (tens of MB), so Colab may take a little while to prepare it.

In [ ]:
import shutil
from google.colab import files

zip_path = shutil.make_archive("/content/f1_data_snapshot", "zip", root_dir="/content", base_dir="data")
print("zipped:", zip_path)
files.download(zip_path)

## Merging back into your local project

Once you have `f1_data_snapshot.zip` downloaded to your computer:
1. Unzip it — you'll get a `data/` folder.
2. Copy `data/raw/*` into your local project's `f1-tyre-degradation/data/raw/` — the folder layout (`data/raw/<year>/r<round>_<code>/`) is identical, so this is a plain merge. Nothing will collide: every session lives in its own uniquely-named folder.
3. Also copy `data/download_errors.csv` and `data/download_summary.csv` if you want the log history, though these aren't required for anything downstream.